# Build a Resilient API Wrapper

In this exercise, you'll combine everything you learned about error handling into a single, production-ready function.

**Your challenge:** Build one function that handles timeouts, fallbacks, and rate limits together.

**Estimated time:** 15-20 minutes

## Setup

Run this cell to import required libraries and initialize the API client.

In [1]:
import os
import time
import random
from dataclasses import dataclass
from openai import OpenAI
from openai import APIError, RateLimitError, APIConnectionError, APITimeoutError

In [2]:
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    OPEN_AI_API_KEY=os.getenv("OPENAI_API_KEY")
    print("OpenAI API key found. Ready to proceed!")
else:
  from google.colab import userdata
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

In [3]:
client = OpenAI(api_key=OPEN_AI_API_KEY)


def make_api_request(prompt: str, model: str = "gpt-4o", max_tokens: int = 150, timeout: float = 30) -> str:
    """
    Helper function to make OpenAI API calls.

    Args:
        prompt: The user prompt to send
        model: The model to use
        max_tokens: Maximum tokens in response
        timeout: Timeout in seconds

    Returns:
        The API response content as a string
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        timeout=timeout
    )
    return response.choices[0].message.content


print("Setup complete.")

Setup complete.


## The Challenge

Build a resilient API wrapper that gracefully handles common failure scenarios:

- **Timeouts** - when the API is slow, retry with longer timeouts
- **Service errors** - when a model is unavailable, fall back to a different model  
- **Rate limits** - when you hit rate limits, back off and retry

Your single function must handle all three scenarios.

### Requirements

1. **Timeout handling**: Start with `initial_timeout`, double it on each retry, up to `max_timeout_retries` attempts
2. **Model fallback**: If the primary model fails with an `APIError`, switch to the fallback model
3. **Rate limit handling**: On `RateLimitError`, use exponential backoff with jitter, up to `max_rate_limit_retries` attempts
4. **Return detailed status**: Return a dataclass with what happened (which model, how many retries, total wait time)

## Part 1: Define the Result Dataclass

First, define a dataclass to hold the result of your resilient API call. This makes it easy for callers to understand what happened.

In [4]:
@dataclass
class APIResult:
    """Result of a resilient API call with detailed status information."""
    success: bool              # Did we eventually get a response?
    content: str               # The response text, or error message if failed
    model_used: str            # Which model produced the response
    attempts: int              # Total number of API call attempts
    total_wait_time: float     # Total time spent waiting (backoff delays)
    used_fallback: bool        # Did we have to fall back to secondary model?

    def __str__(self):
        status = "SUCCESS" if self.success else "FAILED"
        fallback_note = " (fallback)" if self.used_fallback else ""
        return f"[{status}] Model: {self.model_used}{fallback_note}, Attempts: {self.attempts}, Wait: {self.total_wait_time:.2f}s"

## Part 2: Implement the Resilient Wrapper

Now implement the main function. Think about the order of operations:

1. Which errors should trigger a retry vs. a fallback?
2. Should timeout retries happen before or after trying the fallback model?
3. How do you track state across all the different retry scenarios?

In [17]:
def resilient_api_call(
    prompt: str,
    primary_model: str = "gpt-4o",
    fallback_model: str = "gpt-4o-mini",
    initial_timeout: float = 10.0,
    max_timeout_retries: int = 3,
    max_rate_limit_retries: int = 4,
    base_backoff_delay: float = 1.0,
    max_backoff_delay: float = 30.0
) -> APIResult:
    """
    Make a resilient API call that handles timeouts, fallbacks, and rate limits.

    Strategy:
    1. Try the primary model with timeout retries (doubling timeout each time)
    2. If primary model fails with APIError, switch to fallback model
    3. Handle rate limits with exponential backoff + jitter throughout

    Args:
        prompt: The prompt to send
        primary_model: Preferred model to try first
        fallback_model: Backup model if primary fails
        initial_timeout: Starting timeout in seconds
        max_timeout_retries: Max retries for timeout errors (per model)
        max_rate_limit_retries: Max retries for rate limit errors (total)
        base_backoff_delay: Starting delay for exponential backoff
        max_backoff_delay: Maximum delay cap for backoff

    Returns:
        APIResult with success status and metadata about the call
    """
    total_attempts:int = 0
    total_wait_time:float = 0.0
    used_fallback:bool = False
    current_model:str = primary_model
    rate_limit_retries:int = 0
    models:list[str] = [primary_model, fallback_model]
    for model in models:
        timeout = initial_timeout
        timeout_retries =0
        while timeout_retries < max_timeout_retries:
          try:
            total_attempts += 1
            response = make_api_request(prompt, model=model, timeout=timeout)
            return APIResult(
                success=True,
                content=response,
                model_used=model,
                attempts=total_attempts,
                total_wait_time=total_wait_time,
                used_fallback=used_fallback
            )
          except APITimeoutError:
            timeout_retries += 1
            if timeout_retries < max_timeout_retries:
              timeout *= 2
              print(f"Doubling timeout to {timeout:.2f}s")
            else:
              print(f"Max timeout retries reached for {model}")
              break;
          except RateLimitError as re:
            if rate_limit_retries < max_rate_limit_retries:
              delay = min(base_backoff_delay * (2 ** rate_limit_retries), max_backoff_delay)
              # jitter
              delay *= random.uniform(0.5, 1.0)
              time.sleep(delay)
              total_wait_time += delay
              rate_limit_retries += 1
            else:
              return APIResult(
                success=False,
                content= f"Rate limit exceeded :{rate_limit_retries} retries{re}",
                model_used=model,
                attempts=total_attempts,
                total_wait_time=total_wait_time,
                used_fallback=used_fallback
              )
          except APIError as ae:
            print(f"API error (code={getattr(ae,'status_code','unknown')}):{ae}")
            if model == primary_model: # If the primary model failed with APIError
                used_fallback = True  # Mark that we are now using fallback logic
            break # Break from the inner timeout retry loop for the current model, moving to the next model in the outer 'for' loop.
          except Exception as e:
            return APIResult(
                success=False,
                content=f"unexpected error:{type(e).__name__}:{e}",
                model_used=model,
                attempts=total_attempts,
                total_wait_time=total_wait_time,
                used_fallback=used_fallback
            )
    # If the loop finishes, it means all models failed.
    # The last_model_attempted will be the last one in the 'models' list (i.e., the fallback_model).
    return APIResult(
                success=False,
                content="All models failed after maximum retries",
                model_used=models[-1],
                attempts=total_attempts,
                total_wait_time=total_wait_time,
                used_fallback=used_fallback
            )

## Test Your Implementation

Run these tests to verify your implementation handles different scenarios correctly.

In [15]:
# Test 1: Normal operation - should succeed with primary model
print("=" * 50)
print("Test 1: Normal operation")
print("=" * 50)

result = resilient_api_call(
    prompt="What is 2 + 2? Reply with just the number.",
    primary_model="gpt-4o-mini",
    fallback_model="gpt-3.5-turbo",
    initial_timeout=30.0
)

print(result)
print(f"Response: {result.content}")
assert result.success, "Should succeed"
assert not result.used_fallback, "Should not need fallback"
print("PASSED")

Test 1: Normal operation
[SUCCESS] Model: gpt-4o-mini, Attempts: 1, Wait: 0.00s
Response: 4
PASSED


In [18]:
# Test 2: Invalid primary model - should fall back
print("=" * 50)
print("Test 2: Fallback to secondary model")
print("=" * 50)

result = resilient_api_call(
    prompt="What is the capital of France? Reply with just the city name.",
    primary_model="gpt-nonexistent-model",  # This will fail
    fallback_model="gpt-4o-mini",
    initial_timeout=30.0
)

print(result)
print(f"Response: {result.content}")
assert result.success, "Should succeed with fallback"
assert result.used_fallback, "Should have used fallback"
assert result.model_used == "gpt-4o-mini", "Should have used fallback model"
print("PASSED")

Test 2: Fallback to secondary model
API error (code=404):Error code: 404 - {'error': {'message': 'The model `gpt-nonexistent-model` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
[SUCCESS] Model: gpt-4o-mini (fallback), Attempts: 2, Wait: 0.00s
Response: Paris
PASSED


In [19]:
# Test 3: Very short timeout - should retry with longer timeout
print("=" * 50)
print("Test 3: Timeout retry behavior")
print("=" * 50)

result = resilient_api_call(
    prompt="Say hello.",
    primary_model="gpt-4o-mini",
    fallback_model="gpt-3.5-turbo",
    initial_timeout=0.5,  # Will likely timeout
    max_timeout_retries=3
)

print(result)
print(f"Response: {result.content[:100] if result.success else result.content}")
print(f"Total attempts: {result.attempts}")
# This test may or may not succeed depending on network conditions
# The important thing is it should have made multiple attempts
print(f"Made multiple attempts: {result.attempts > 1}")

Test 3: Timeout retry behavior
[SUCCESS] Model: gpt-4o-mini, Attempts: 1, Wait: 0.00s
Response: Hello! How can I assist you today?
Total attempts: 1
Made multiple attempts: False


In [20]:
# Test 4: General usage
print("=" * 50)
print("Test 4: General usage")
print("=" * 50)

result = resilient_api_call(
    prompt="Explain what an API is in one sentence.",
    primary_model="gpt-4o-mini",
    fallback_model="gpt-3.5-turbo",
    initial_timeout=30.0
)

print(result)
print(f"\nResponse:\n{result.content}")

Test 4: General usage
[SUCCESS] Model: gpt-4o-mini, Attempts: 1, Wait: 0.00s

Response:
An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and interact with each other.
